# Install packages

# Import

In [1]:
import os
import time
import torch
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import QM9, ZINC
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

/hdfs1/Data/weather/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libm.so.6: version `GLIBC_2.29' not found (required by /hdfs1/Data/weather/.venv/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/hdfs1/Data/weather/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libm.so.6: version `GLIBC_2.29' not found (required by /hdfs1/Data/weather/.venv/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


device(type='cuda')

# Model

In [2]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class Regress_graph(torch.nn.Module):
    def __init__(self, num_layer, num_feature, num_hidden):
        super(Regress_graph, self).__init__()
        self.num_layers = num_layer
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_feature, num_hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(num_hidden, num_hidden))
        self.lt1 = torch.nn.Linear(num_hidden, 1)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, gc):
        x, edge_index, batch = gc.x, gc.edge_index, gc.batch
        x = x.float()
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = global_mean_pool(x, batch)
        x = self.lt1(x)
        return x

# Utils

In [3]:
def train_test_val_split(dataset, shuffle=True):
    N = len(dataset)
    if shuffle:
        idx = torch.randperm(N)
    else:
        idx = torch.arange(N)
    train = []
    val = []
    test = []
    for i in range(N):
        if i < N//2:
            train.append(dataset[idx[i]])
        elif i < 3*N//4 and i >= N//2:
            val.append(dataset[idx[i]])
        else:
            test.append(dataset[idx[i]])
    return train, test, val

In [4]:
def train_model(train_loader, model, loss_fn, optimizer, property=None):
  all_output_train = torch.tensor([]).to(device)
  all_labels_train = torch.tensor([]).to(device)
  train_loss = 0
  model.train()
  optimizer.zero_grad()

  for graphs in train_loader:
    graphs = graphs.to(device)
    out = model(graphs) #.astype(float)
    # print(out.shape, graphs.y.shape) # graphs.y[:, 0].view(-1, 1).shape
    loss = loss_fn(out, graphs.y.view(-1, 1).to(device))
    train_loss += loss.item()
    all_output_train = torch.cat((all_output_train, out))
    if property is not None:
        all_labels_train = torch.cat((all_labels_train, graphs.y[:, property].view(-1, 1).to(device)))
    else:
        all_labels_train = torch.cat((all_labels_train, graphs.y.to(device)))
    loss.backward()
    optimizer.step()

  train_loss = train_loss / len(train_loader)

  return train_loss / torch.std(all_labels_train).item()

def infer_model(loader, model, loss_fn, property=None):
  all_output = torch.tensor([]).to(device)
  all_labels = torch.tensor([]).to(device)
  all_loss = 0
  model.eval()

  for graphs in loader:
    graphs = graphs.to(device)
    out = model(graphs)
    loss = loss_fn(out, graphs.y.view(-1, 1).to(device))
    all_loss += loss.item()
    all_output = torch.cat((all_output, out))
    if property is not None:
        all_labels = torch.cat((all_labels, graphs.y[:, property].view(-1, 1).to(device)))
    else:
        all_labels = torch.cat((all_labels, graphs.y.to(device)))

  all_loss = all_loss / len(loader)

  return all_loss  / torch.std(all_labels).item()

# Main

In [5]:
dataset = ZINC(root='../dataset/ZINC', subset=True)
train_split, test_split, val_split = train_test_val_split(dataset, shuffle=True)
train_loader = DataLoader(train_split, batch_size=128, shuffle=True)
val_loader = DataLoader(val_split, batch_size=128, shuffle=False)
test_loader = DataLoader(test_split, batch_size=128, shuffle=False)

num_layer = 2
num_feature = dataset[0].x.shape[1]
num_hidden = 512
# property = 6
model = Regress_graph(num_layer, num_feature, num_hidden).to(device)
loss_fn = torch.nn.L1Loss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

/hdfs1/Data/weather/.venv/lib/python3.10/site-packages/torch_geometric/data/dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.load(f) 

In [6]:
s = 0
for i in range(10000):
    s += dataset[i].y.item()
print(s/10000)

0.015297334436047822


In [7]:
dataset[0].y.item()

0.8350355625152588

In [8]:
if not os.path.exists("../save"):
  os.mkdir("../save")
if not os.path.exists("../save/graph_reg"):
  os.mkdir("../save/graph_reg")
if not os.path.exists("../save/graph_reg/baselines"):
  os.mkdir("../save/graph_reg/baselines")

best_val_loss = float('inf')
best_test_loss = float('inf')
best_val_acc = 0
best_test_acc = 0
all_train_loss = []
all_val_loss = []
all_test_loss = []
for epoch in tqdm(range(300)):
  #Train model
  train_loss = train_model(train_loader, model, loss_fn, optimizer)
  all_train_loss.append(train_loss)
  #Validate Model
  val_loss = infer_model(val_loader, model, loss_fn)
  all_val_loss.append(val_loss)
  #Test Model
  test_loss = infer_model(test_loader, model, loss_fn)
  all_test_loss.append(test_loss)
  #save model
  if val_loss <= best_val_loss or epoch == 0:
    best_val_loss = val_loss
    best_test_loss = test_loss
    torch.save(model.state_dict(), '../save/graph_reg/baselines/baseline_ZINC_subset.pt')
    print("\n")
    print(f"train loss: {train_loss}")
    print(f"val loss: {val_loss}")
    print(f"test loss: {test_loss}")
    print("Best model saved")
    print("\n")

  if epoch == 0 or epoch%25 == 0:
    print("\n")
    print(f"train loss: {train_loss}")
    print(f"val loss: {val_loss}")
    print(f"test loss: {test_loss}")
    print("\n")


print("\n")
print(f"Best Val Loss: {best_val_loss}")
print(f"Best Test Loss: {best_test_loss}")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 1/300 [00:02<11:29,  2.31s/it]



train loss: 8.239789739842216
val loss: 13.020928414401919
test loss: 14.093038664051175
Best model saved




train loss: 8.239789739842216
val loss: 13.020928414401919
test loss: 14.093038664051175




  1%|▏         | 4/300 [00:05<06:42,  1.36s/it]



train loss: 33.80558606897928
val loss: 10.308452585477992
test loss: 11.164348796412293
Best model saved




...



train loss: 51.9329700443891
val loss: 35.26814193528961
test loss: 37.97557296280625




 92%|█████████▏| 276/300 [05:50<00:31,  1.32s/it]



train loss: 48.79621777475861
val loss: 58.546391928958364
test loss: 63.15394853239715




100%|██████████| 300/300 [06:22<00:00,  1.27s/it]



Best Val Loss: 0.7155530582618486
Best Test Loss: 0.74536150409114
